# Intro til Machine Learning — 1: Datamanipulation 📊

Velkommen til jeres første skridt ind i machine learning! Inden en computer kan *lære*
noget som helst, skal den have noget at lære **fra** — nemlig data. Og data i den virkelige
verden er rodet, fuld af huller og aldrig helt som man vil have det. I denne notebook lærer
I at tæmme et rigtigt datasæt med værktøjet **pandas** — og som forsøgskaniner har vi valgt...
**Pokémon** 🔥💧🌿

> **Sådan bruger I notebooks'ene i dette emne:**
> - Kør cellerne fra toppen og nedad. Læs teksten, kør eksemplerne, og lav opgaverne i `Opgaver`-afsnittene.
> - I skal **ikke** skrive kode fra bunden — I skal *ændre* i kode, der allerede virker, og se hvad der sker. Det er sådan, man forstår tingene.
> - Der er med vilje **flere opgaver, end I kan nå**. Ingen panik — ⭐-opgaver er til jer, der er foran.
> - Opgaver med 🐛 indeholder en **bevidst fejl**, som I skal finde og rette. Så hvis en 🐛-celle fejler, er det meningen!

**Emnets køreplan:** 1) Datamanipulation *(I er her)* → 2) PyTorch & gradient descent → 3) Neurale netværk → 4) Aktiveringsfunktioner → 5) MNIST: netværket der kan læse håndskrift.

# 1: Data — råstoffet i machine learning

I har lige lært at programmere: variabler, `if`-sætninger, `for`-løkker, funktioner, numpy,
matplotlib, klasser og f-strings. Indtil nu har **I** fortalt computeren præcis, hvad den skal gøre, linje for linje.
Machine learning vender det på hovedet: i stedet for at skrive reglerne selv, viser vi computeren
en masse **eksempler** og lader den selv finde reglerne. 🤯

Et datasæt er (næsten altid) en tabel:
- Hver **række** er én observation — én patient, én kunde... eller én Pokémon.
- Hver **kolonne** er en egenskab ved observationen — alder, pris... eller angrebsstyrke. På ML-sprog kaldes kolonnerne **features**.

Før vi overhovedet må røre et neuralt netværk, skal vi kunne indlæse, undersøge, rense og
omforme sådan en tabel. Det er ikke det mest glamourøse i ML — men det er dét, folk der
arbejder med ML, bruger *størstedelen* af deres tid på. Så lad os blive gode til det!

## Setup

Først installerer vi `kagglehub` — et lille bibliotek, der kan hente datasæt fra
[Kaggle](https://www.kaggle.com), som er internettets største samlingssted for datasæt.
(`!` foran en linje betyder "kør dette i terminalen i stedet for i Python".)

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)  # gør "tilfældige" tal ens for os alle sammen — mere om det senere

## Hent datasættet

Nu henter vi Pokémon-datasættet fra Kaggle. `kagglehub.dataset_download` downloader
datasættet (første gang) og fortæller os, hvilken mappe det ligger i. Derefter indlæser
`pd.read_csv` CSV-filen som en **DataFrame** — pandas' ord for en tabel.

In [ ]:
sti = kagglehub.dataset_download("abcsds/pokemon")
print("Datasættet ligger her:", sti)

df = pd.read_csv(sti + "/Pokemon.csv")
df.head()

> 🚨 **Plan B:** Hvis Kaggle-cellen ovenfor fejler (fx fordi camp-wifi har en dårlig dag),
> så fjern `#`'erne i cellen nedenunder og kør den i stedet. Den henter en kopi af præcis
> samme fil fra vores GitHub.

In [ ]:
# 🚨 Plan B — kør KUN denne celle, hvis Kaggle-cellen ovenfor fejlede:
# df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/Pokemon.csv")
# df.head()

## Hvad kigger vi på?

`df.head()` viser de første 5 rækker. Kolonnerne betyder:

| Kolonne | Betydning |
|---|---|
| `#` | Pokémonens nummer i Pokédex'en (bemærk: Mega-udgaver deler nummer) |
| `Name` | Navnet |
| `Type 1` / `Type 2` | Typen(-erne) — fx Grass, Fire, Water. Ikke alle har en Type 2! |
| `Total` | Summen af de seks stats nedenfor — et groft mål for samlet styrke |
| `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` | De seks kamp-stats |
| `Generation` | Hvilken spilgeneration (1–6) den kommer fra |
| `Legendary` | `True` hvis den er legendarisk 🌟 |

**Én vigtig detalje:** flere kolonnenavne indeholder mellemrum (`Sp. Atk`, `Type 1`).
Derfor skriver vi altid `df["Sp. Atk"]` med kantede parenteser og anførselstegn — det virker for alle kolonnenavne, altid.

### Opgaver

##### Opgave 1.1
Kør download-cellen ovenfor (hvis I ikke allerede har gjort det) og kig på stien, der bliver
printet. Åbn filpanelet i venstre side af Colab (📁-ikonet) og se, om I kan finde `Pokemon.csv`.
Hvor mange **rækker** og **kolonner** har tabellen? (Psst: kør cellen nedenunder.)

In [ ]:
df.shape  # (rækker, kolonner)

##### Opgave 1.2
Kig på tabellen fra `df.head()`. Hvilke kolonner indeholder **tal**, og hvilke indeholder
**tekst eller sandt/falsk**? Hvorfor er det vigtigt at vide, når en computer skal regne på
dataene?

*Skriv jeres svar her:* $\dots$

##### Opgave 1.3
`df.head()` kan tage et tal i parentesen. Prøv at ændre tallet i cellen nedenfor — hvad gør det?
Og hvad gør `df.tail()` mon?

In [ ]:
df.head()  # prøv fx df.head(10) — og bagefter df.tail(3)

# 2: Pandas — jeres nye superkraft

En DataFrame er som et regneark med turbo på: I kan udvælge, filtrere, sortere og regne på
hundredtusindvis af rækker med én linje kode. Lad os tage værktøjerne ét ad gangen.

## Udvælg kolonner

Én kolonne med `df["Name"]` — flere kolonner med **dobbelte** kantede parenteser
(fordi man giver pandas en *liste* af kolonnenavne):

In [ ]:
df[["Name", "HP", "Speed"]].head()

## Udvælg rækker

`df.iloc[i]` giver række nummer `i` (regnet fra 0 — som altid i Python), og man kan slice
ligesom i numpy:

In [ ]:
df.iloc[0]        # den allerførste Pokémon (gæt hvem 😉)

In [ ]:
df.iloc[10:13]    # række 10, 11 og 12

## Boolean-masker: stil hele tabellen ét spørgsmål

Det her er pandas' fedeste trick. `df["HP"] > 150` spørger *alle 800 rækker på én gang*
"er din HP over 150?" og svarer med 800 gange `True`/`False`. Putter man svaret ind i
`df[...]`, får man kun rækkerne med `True`:

In [ ]:
df["HP"] > 150            # 800 svar: True eller False

In [ ]:
df[df["HP"] > 150]        # kun rækkerne hvor svaret var True

## Kombinér spørgsmål — og HUSK parenteserne!

Flere betingelser kombineres med `&` (og) samt `|` (eller). **Hver betingelse SKAL stå i
sin egen parentes** — ellers bliver Python forvirret og kaster en lang, sur fejlbesked
(det kommer I til at opleve i opgave 2.4 😈):

In [ ]:
hurtige_taenks = df[(df["Defense"] > 100) & (df["Speed"] > 100)]
hurtige_taenks[["Name", "Defense", "Speed"]].head()

## Sortér, tæl og få overblik

- `sort_values("kolonne")` sorterer — `ascending=False` giver størst først.
- `value_counts()` tæller, hvor mange gange hver værdi optræder.
- `describe()` giver statistik (middelværdi, spredning, min/max...) for alle talkolonner på én gang.

In [ ]:
# Hvem er hurtigst i hele Pokédex'en?
df.sort_values("Speed", ascending=False)[["Name", "Speed"]].head()

In [ ]:
df["Type 1"].value_counts()

In [ ]:
df.describe()

### Opgaver

##### Opgave 2.1
Koden nedenfor viser de **første 5** rækker. Ændr den, så den viser de **sidste 10**.

In [ ]:
df.head()

##### Opgave 2.2
Koden viser `Name` og `HP`. Ændr den, så den viser `Name`, `Attack` og `Speed` — sorteret
efter `Attack` med den største først.

In [ ]:
df[["Name", "HP"]].head()

##### Opgave 2.3
Udfyld boolean-masken, så I kun ser Pokémon med `Speed` over 100.

In [ ]:
hurtige = df[df[...] > ...]
hurtige[["Name", "Speed"]].head()

##### Opgave 2.4 🐛
Koden nedenfor SKULLE finde Pokémon, der både har `HP` over 100 **og** `Speed` over 100...
men den crasher med en lang fejlbesked. Læs det sidste linje af fejlbeskeden — og ret så
koden (hint: afsnittet om parentes-reglen).

In [ ]:
staerke_og_hurtige = df[df["HP"] > 100 & df["Speed"] > 100]
staerke_og_hurtige[["Name", "HP", "Speed"]]

##### Opgave 2.5
Cellen tæller, hvor mange Pokémon der findes af hver `Type 1`. Prøv også med `Type 2` og
`Generation`. Hvilken type er den mest almindelige — og hvilken er den mest sjældne?

In [ ]:
df["Type 1"].value_counts()

##### Opgave 2.6
Brug en boolean-maske til at finde **Pikachu** og se dens stats. Udfyld navnet i masken.
Er Pikachu egentlig så sej, som serien påstår? ⚡

In [ ]:
df[df["Name"] == ...]

##### Opgave 2.7
Kør `describe()`-cellen ovenfor igen og kig på rækken **std** (standardafvigelsen — et mål
for hvor *spredte* værdierne er). Hvilken af de seks kamp-stats har den største spredning?
Hvad betyder det i praksis?

*Skriv jeres svar her:* $\dots$

##### Opgave 2.8 ⭐
Find de 5 stærkeste **ikke-legendariske** Pokémon (målt på `Total`). I skal kombinere tre
ting i én pipeline: en maske der frasorterer legendariske (`==False` eller `~`), en sortering
og en `head`. Start fra koden nedenfor.

In [ ]:
# Byg videre på denne — lige nu viser den bare de 5 stærkeste OVERHOVEDET
df.sort_values("Total", ascending=False)[["Name", "Total", "Legendary"]].head()

##### Opgave 2.9 🐛
Koden nedenfor skulle vise de stærkeste Pokémon ØVERST... men tabellen er stadig i den gamle
rækkefølge, og de svageste ligger først når man endelig sorterer. Der er **to** problemer —
find og ret dem begge.

In [ ]:
df.sort_values("Total")
df.head()

##### Opgave 2.10 ⭐
Hvor mange **legendariske** Pokémon er der i hver generation? Udfyld koden: filtrér først
til kun legendariske, og tæl så med `value_counts()` på `Generation`.

In [ ]:
legendariske = df[df[...] == ...]
legendariske[...].value_counts()

# 3: Datarensning — den virkelige verden er rodet

Rigtige datasæt har huller: en sensor der svigtede, et spørgeskema-felt der ikke blev udfyldt,
en Pokémon der bare ikke *har* nogen anden type. Manglende værdier hedder `NaN`
(*Not a Number*), og de skal håndteres, før man kan lave ML — ellers brokker modellerne sig.

`isna()` spørger hver celle "mangler du?", og `.sum()` tæller op pr. kolonne:

In [ ]:
df.isna().sum()

386 Pokémon mangler `Type 2` — det er ikke en fejl i data, de har bare kun én type!

Der er to klassiske strategier:

1. **`fillna(værdi)`** — udfyld hullerne med noget fornuftigt.
2. **`dropna()`** — smid alle rækker med huller ud. Hurtigt, men brutalt: man mister data!

Ligesom `sort_values` ændrer de ikke den oprindelige tabel — de returnerer en ny, som man
selv skal gemme:

In [ ]:
df_udfyldt = df.fillna("Ingen")
df_udfyldt.head()   # kig på Type 2 for fx Charmander

In [ ]:
df_uden_huller = df.dropna()
print("Før: ", df.shape)
print("Efter:", df_uden_huller.shape)   # av — der røg nogle rækker!

## Nye kolonner: regn på hele tabellen på én gang

Man kan lave en ny kolonne ud fra de gamle med almindelig matematik — pandas regner så
for alle 800 rækker samtidig:

In [ ]:
df["Offensiv"] = df["Attack"] + df["Sp. Atk"]
df.sort_values("Offensiv", ascending=False)[["Name", "Attack", "Sp. Atk", "Offensiv"]].head()

## groupby: del op i hold, og regn pr. hold

`groupby("Type 1")` deler tabellen op i ét hold pr. type. Derefter vælger man en kolonne og
en udregning — fx gennemsnittet af `Attack` pr. type:

In [ ]:
df.groupby("Type 1")["Attack"].mean().sort_values(ascending=False)

## Fra sandt/falsk til tal

ML-modeller spiser kun tal. Kolonnen `Legendary` indeholder `True`/`False`, men med
`astype(int)` bliver det til 1/0 — klar til en model:

In [ ]:
df["Legendary_tal"] = df["Legendary"].astype(int)
df[["Name", "Legendary", "Legendary_tal"]].tail()   # de sidste rækker er legendariske

### Opgaver

##### Opgave 3.1
Kør `df.isna().sum()` igen. Hvorfor giver det god mening, at det netop er `Type 2`, der
mangler værdier — og hvad *betyder* et hul i den kolonne egentlig?

*Skriv jeres svar her:* $\dots$

##### Opgave 3.2
Koden bruger `fillna("Ingen")`. Skift til `dropna()` og sammenlign `shape` før og efter.
Præcis hvor mange Pokémon mistede I — og hvilke Pokémon er det, der ryger ud?

In [ ]:
ny_df = df.fillna("Ingen")
print("Før: ", df.shape)
print("Efter:", ny_df.shape)

##### Opgave 3.3
Lav kolonnen `Angreb_pr_Forsvar` som `Attack` divideret med `Defense` — udfyld formlen.
En høj værdi betyder "kanon som kan skyde, men er lavet af glas" 🥃. Hvilken Pokémon er
datasættets største glaskanon?

In [ ]:
df["Angreb_pr_Forsvar"] = df["Attack"] / ...
df.sort_values("Angreb_pr_Forsvar", ascending=False)[["Name", "Attack", "Defense", "Angreb_pr_Forsvar"]].head()

##### Opgave 3.4
`Offensiv`-kolonnen fik vi lavet ovenfor. Lav på samme måde en `Defensiv`-kolonne
(`Defense` + `Sp. Def`) og find den mest defensive Pokémon i hele datasættet.

In [ ]:
df["Offensiv"] = df["Attack"] + df["Sp. Atk"]
df.sort_values("Offensiv", ascending=False)[["Name", "Offensiv"]].head()

##### Opgave 3.5 🐛
Nogen ville regne gennemsnittet af angrebs-statten, men pandas kaster en `KeyError`.
Kør cellen, læs fejlbeskeden, og ret fejlen. (Hint: `df.columns` viser alle kolonnenavne,
præcis som de er stavet.)

In [ ]:
df["attack"].mean()

##### Opgave 3.6
Udfyld `groupby`-koden, så I finder den type (`Type 1`), der har det højeste
**gennemsnitlige** `Attack`.

In [ ]:
df.groupby(...)[...].mean().sort_values(ascending=False)

##### Opgave 3.7
Er der *power creep* i Pokémon? Brug `groupby` på `Generation` og tag gennemsnittet af
`Total`. Bliver Pokémon stærkere for hver generation — hvad siger tallene?

In [ ]:
df.groupby("Generation")["Total"].mean()

##### Opgave 3.8 ⭐
Lav en kolonne `Hurtigere_end_staerk`, der er `True`, når `Speed` er større end `Attack`.
Brug derefter `.mean()` på kolonnen til at finde ud af, hvor stor en **andel** af alle
Pokémon det gælder for. (Hvorfor giver `.mean()` på en True/False-kolonne en andel? Tænk
over, hvad `True` er som tal...)

In [ ]:
df["Hurtigere_end_staerk"] = ...
df["Hurtigere_end_staerk"].mean()

##### Opgave 3.9
Forestil jer, at tabellen ikke var Pokémon, men **patienter på et hospital**, og at kolonnen
med huller var "resultat af blodprøve". Hvorfor kan det være direkte farligt bare at køre
`dropna()` og smide alle rækker med huller ud?

*Skriv jeres svar her:* $\dots$

# 4: Plots og tal-gymnastik — gør data klar til ML

Tal i en tabel er svære at *fornemme*. Et godt plot afslører på ét sekund mønstre, som man
kunne stirre på tallene i timevis uden at se. Og til sidst i dette afsnit lærer I det
vigtigste ML-forberedelsestrick af dem alle: **standardisering**.

## Broen tilbage til numpy

I kender allerede numpy. En pandas-kolonne kan når som helst omdannes til et numpy-array
med `.values` — så alt, hvad I har lært om numpy, virker stadig:

In [ ]:
hp = df["HP"].values
print(type(hp))
print(hp[:10])
print("gennemsnit:", hp.mean().round(1))

## Histogrammer: hvordan fordeler en stat sig?

In [ ]:
plt.hist(df["HP"], bins=30)
plt.title("Fordeling af HP")
plt.xlabel("HP")
plt.ylabel("Antal Pokémon")
plt.show()

## Scatterplots: hænger to stats sammen?

In [ ]:
plt.scatter(df["Attack"], df["Defense"], alpha=0.5)
plt.title("Attack mod Defense")
plt.xlabel("Attack")
plt.ylabel("Defense")
plt.show()

## Farvelæg grupper

Kalder man `plt.scatter` to gange, ender begge grupper i samme figur — og med `label` +
`plt.legend()` kan man se, hvem der er hvem. Kan man *se* de legendariske i punktskyen?

In [ ]:
almindelige = df[df["Legendary"] == False]
legendariske = df[df["Legendary"] == True]

plt.scatter(almindelige["Attack"], almindelige["Defense"], alpha=0.4, label="Almindelig")
plt.scatter(legendariske["Attack"], legendariske["Defense"], color="red", label="Legendarisk 🌟")
plt.title("Attack mod Defense")
plt.xlabel("Attack")
plt.ylabel("Defense")
plt.legend()
plt.show()

## Søjlediagrammer af optællinger

`value_counts()` fra afsnit 2 kan plottes direkte:

In [ ]:
antal = df["Type 1"].value_counts()
plt.bar(antal.index, antal.values)
plt.title("Antal Pokémon pr. type")
plt.xticks(rotation=45, ha="right")   # drej navnene så de kan læses
plt.ylabel("Antal")
plt.show()

## Korrelation: hvor meget hænger stats sammen?

**Korrelation** måler, hvor meget to kolonner følges ad — fra −1 (modsat) over 0 (intet
mønster) til +1 (helt i takt). `df.corr()` regner den for alle par af talkolonner, og med
`plt.imshow` kan vi tegne resultatet som et farvekort:

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Total"]
korr = df[stats].corr(numeric_only=True)

plt.imshow(korr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="korrelation")
plt.xticks(range(len(stats)), stats, rotation=45, ha="right")
plt.yticks(range(len(stats)), stats)
plt.title("Korrelation mellem stats")
plt.show()

## Standardisering — det vigtigste trick før ML 🎯

Kig på tallene: `HP` ligger typisk mellem 20 og 150, men forestil jer et datasæt med både
"alder" (0–100) og "årsløn" (0–1.000.000). En ML-model, der får begge tal råt ind, vil tro,
at løn er 10.000 gange vigtigere end alder — bare fordi tallene er større!

Løsningen er at **standardisere** hver kolonne, så den får gennemsnit 0 og spredning 1:

$$x_{\text{ny}} = \frac{x - \text{gennemsnit}}{\text{spredning}}$$

Efter standardisering betyder $x_{\text{ny}} = 2$ altså "2 spredninger over gennemsnittet" —
uanset om kolonnen var HP eller årsløn. Alle kolonner taler pludselig samme sprog.

**Gem denne celle i baghovedet** — i notebook 2 og 3 skal I se, hvad der sker med et neuralt
netværk, når man *glemmer* at standardisere. Det er ikke kønt. 💀

In [ ]:
attack = df["Attack"].values
attack_std = (attack - attack.mean()) / attack.std()

fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].hist(attack, bins=30)
akser[0].set_title("Attack — rå tal")
akser[1].hist(attack_std, bins=30, color="orange")
akser[1].set_title("Attack — standardiseret")
plt.show()

print("Før:  gennemsnit =", attack.mean().round(2), " spredning =", attack.std().round(2))
print("Efter: gennemsnit =", attack_std.mean().round(2), " spredning =", attack_std.std().round(2))

Bemærk: histogrammet har **præcis samme facon** — vi har ikke ødelagt nogen information,
kun flyttet og skaleret tallene, så de er nemme for en model at arbejde med.

### Opgaver

##### Opgave 4.1
Histogram-koden viser fordelingen af `HP`. Skift kolonnen til `Speed`, og prøv `bins=10`
og `bins=60`. Hvad gør `bins`? Og hvordan vil I beskrive fordelingen af Speed med ét ord?

In [ ]:
plt.hist(df["HP"], bins=30)
plt.title("Fordeling af HP")
plt.xlabel("HP")
plt.ylabel("Antal Pokémon")
plt.show()

##### Opgave 4.2
Udfyld scatterplottet, så det viser `Sp. Atk` mod `Sp. Def` — med aksetitler!

In [ ]:
plt.scatter(df[...], df[...], alpha=0.5)
plt.xlabel(...)
plt.ylabel(...)
plt.show()

##### Opgave 4.3
Tag det farvelagte scatterplot (legendariske vs. almindelige) og lav det om: prøv andre
farver, en anden `alpha`, og en markørstørrelse med fx `s=80`. Hvor "gemmer" de
legendariske sig i plottet — og tror I, en model ville kunne finde dem ud fra stats alene?

In [ ]:
almindelige = df[df["Legendary"] == False]
legendariske = df[df["Legendary"] == True]

plt.scatter(almindelige["Attack"], almindelige["Defense"], alpha=0.4, label="Almindelig")
plt.scatter(legendariske["Attack"], legendariske["Defense"], color="red", label="Legendarisk 🌟")
plt.legend()
plt.show()

##### Opgave 4.4 🐛
Plottet nedenfor påstår at vise "Speed mod HP" — men der er rod i det: akserne passer ikke
til titlerne, og forklaringsboksen mangler. Ret koden, så plot og tekst passer sammen.

In [ ]:
plt.scatter(df["HP"], df["Speed"], alpha=0.5, label="Pokémon")
plt.title("Speed (x) mod HP (y)")
plt.xlabel("Speed")
plt.ylabel("HP")
plt.show()

##### Opgave 4.5
Udfyld søjlediagrammet, så det viser antallet af Pokémon pr. **generation** i stedet for
pr. type.

In [ ]:
antal = df[...].value_counts()
plt.bar(antal.index, antal.values)
plt.title(...)
plt.ylabel("Antal")
plt.show()

##### Opgave 4.6
Kør korrelations-farvekortet igen og kig godt efter (ignorér diagonalen — alt korrelerer
100 % med sig selv). Hvilke to *forskellige* stats hænger stærkest sammen? Og hvorfor er
`Total`-rækken så rød hele vejen?

*Skriv jeres svar her:* $\dots$

##### Opgave 4.7
Udfyld standardiserings-formlen for `Speed`-kolonnen, og tjek at det nye gennemsnit er
(meget tæt på) 0 og spredningen (meget tæt på) 1.

In [ ]:
speed = df["Speed"].values
speed_std = (speed - ...) / ...

print("gennemsnit:", speed_std.mean().round(4))
print("spredning: ", speed_std.std().round(4))

##### Opgave 4.8 ⭐
Skabelonen nedenfor laver 1×2 subplots. Udvid den til et 2×2-grid med **fire selvvalgte
plots** af Pokémon-data — fx et histogram, et scatterplot, et søjlediagram og ét I selv
finder på. Husk titler, så man kan se, hvad man kigger på!

In [ ]:
fig, akser = plt.subplots(1, 2, figsize=(11, 4))
akser[0].hist(df["HP"], bins=30)
akser[0].set_title("HP")
akser[1].scatter(df["Attack"], df["Defense"], alpha=0.4)
akser[1].set_title("Attack mod Defense")
plt.tight_layout()
plt.show()

# Godt gået! 🎉

I kan nu indlæse, undersøge, filtrere, rense, gruppere, plotte og standardisere et rigtigt
datasæt. Det lyder måske ikke som "machine learning" — men det er fundamentet under ALT,
hvad der kommer nu.

**Næste stop:** `2-PyTorch-og-gradient-descent.ipynb`, hvor I møder PyTorch — biblioteket,
der driver en kæmpe del af moderne AI — og lærer computeren at finde den bedste linje
*helt selv*. Pokémonerne kommer med. ⚡